# Hypothesis Testing（仮説検証）

## 目的
EDAの結果、2026年は60日以内募集枠充足率が大きく低下しており、
特に「応募 → 推薦」「推薦 → 職場見学」の選考前半で転換率が悪化していることを確認した。

また、その背景要因として、
- スキル・経験のミスマッチ
- 給与条件のミスマッチ
- 応募意思確認から推薦までのリードタイム長期化
が候補として考えられる。

本Notebookでは、これらの要因と推薦結果・選考途中辞退との関係を統計的に検証し、
2026年に見られた推薦率低下や選考途中辞退の増加と関連する要因を整理する。

EDAでは選考前半の転換率悪化が主な課題として確認された一方、選考後半では辞退率に大きな年次差は見られなかった。
そのため、辞退に関する分析では特定の工程に限定せず、まず選考途中辞退全体を対象として、給与条件や選考リードタイムとの関係を確認する。

なお、本分析は観察データを用いた分析であるため、
統計的な関連が確認された場合でも因果関係を直接示すものではない。


## 検証する仮説

### 仮説1：スキル・経験のミスマッチ
求人が求める経験・スキルと応募者の保有経験・スキルの乖離が大きいほど、
推薦されにくいのではないか。

### 仮説2：給与条件のミスマッチ
応募者の希望給与が求人給与を上回るほど、辞退が発生しやすいのではないか。
あわせて、推薦との関係も確認する。

### 仮説3：選考リードタイム
応募意思確認から推薦までの日数が長いほど、
選考途中での辞退が発生しやすいのではないか。


## 分析の進め方

**傾向の確認**  
集計やグループ間比較により、各要因と推薦結果・選考途中辞退の傾向を確認する。

**統計的な検証**  
カイ二乗検定やMann-Whitney U検定を用いて関連・差を検証し、
p値とあわせて効果量も確認する。

**複数要因を考慮した確認**  
ロジスティック回帰分析を用いて複数要因を同時に考慮し、
各要因と推薦結果・選考途中辞退との関連を総合的に確認する。

以上をもとに、EDAで確認した2026年の選考状況悪化について、どの要因との関連が強いかを整理する。

## 1. ライブラリの読み込み
分析に必要なPythonライブラリを読み込む。

- `sqlite3`：SQLiteデータベースへ接続する
- `pandas`：表形式データをDataFrameとして扱う
- `numpy`：数値計算を行う
- `matplotlib`：グラフを作成する
- `scipy.stats`：統計検定（カイ二乗検定 / Mann-WhitneyのU検定）を行う
- `statsmodels`：統計モデルを用いた分析（ロジスティック回帰分析）を行う

In [164]:
import sqlite3

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, mannwhitneyu
import statsmodels.formula.api as smf

## 2. SQLiteデータベースへの接続
Pythonから、合成データおよびSQL分析マートを格納したSQLiteデータベースへ接続する。

In [165]:
conn = sqlite3.connect(
    "../database/recruitment_analytics.sqlite"
)

### 接続先データベースの確認
想定したSQLiteファイルへ接続できていることを確認する。

In [166]:
pd.read_sql_query(
    "PRAGMA database_list;",
    conn
)

,seq,name,file
0,0,main,c:\Users\tomoe\OneDrive\PERSOL\キャリアスカウト\recrui...


## 3. 分析マートの読み込み
SQLで作成した2つの分析マートを、pandasのDataFrameとしてPythonへ読み込む。

- `job_df`：求人単位の分析データ
- `application_df`：応募案件単位の分析データ

In [167]:
job_df = pd.read_sql_query(
    """
    SELECT *
    FROM mart_job_analysis
    """,
    conn
)

application_df = pd.read_sql_query(
    """
    SELECT *
    FROM mart_application_analysis
    """,
    conn
)

## 4. データ件数と内容の確認
読み込んだDataFrameについて、行数・列数と先頭データを確認する。
※SQLiteから想定したデータを正常に読み込めていることを確認する。

In [168]:
print("job_df:", job_df.shape)
print("application_df:", application_df.shape)

### 求人マートの先頭行
display(job_df.head())

### 応募案件マートの先頭行
display(application_df.head())

job_df: (800, 14)
application_df: (2658, 26)


,job_id,open_year,open_month,occupation_name,required_slots,offered_hourly_wage,required_skill_level,work_style,job_60d_observed_flag,entry_count,application_count,placement_count,placement_60d_count,fill_rate_60d
0,JOB00001,2025,2025-10,コールセンター,1,1550,3,onsite,1,0,1,0,0.0,0.000
1,JOB00002,2025,2025-10,コールセンター,3,1500,2,hybrid,1,1,4,0,0.0,0.000
2,JOB00003,2025,2025-12,データ入力,3,1250,1,onsite,1,5,12,3,1.0,0.333
3,JOB00004,2025,2025-12,DM（データマネジメント）,1,2200,3,onsite,1,6,3,1,1.0,1.000
4,JOB00005,2025,2025-05,DM（データマネジメント）,1,2250,4,hybrid,1,1,5,1,0.0,0.000


,application_id,job_id,application_source,withdrawal_stage,application_status,application_year,application_month,application_30d_observed_flag,wage_gap,wage_gap_rate,...,recommended_flag,screened_out_flag,confirmed_flag,withdrawal_flag,process_withdrawal_flag,visit_completed_flag,recommend_to_schedule_days,schedule_to_visit_days,placed_flag,intent_to_decision_days
0,APP000785,JOB00001,ca_introduction,NaN,rejected,2025,2025-10,1,-150,-0.088,...,1,0,0,0,0,1,5.0,5.0,0,NaN
1,APP000965,JOB00002,ca_introduction,post_visit,withdrawn,2025,2025-11,1,0,0.000,...,1,0,0,1,0,1,4.0,5.0,0,NaN
2,APP001060,JOB00002,self_entry,NaN,rejected,2025,2025-12,1,0,0.000,...,1,0,0,0,0,1,5.0,4.0,0,NaN
3,APP001245,JOB00002,ca_introduction,NaN,rejected,2026,2026-01,1,-350,-0.189,...,1,0,0,0,0,0,NaN,NaN,0,NaN
4,APP002034,JOB00002,ca_introduction,NaN,rejected,2026,2026-06,1,-450,-0.231,...,1,0,0,0,0,0,NaN,NaN,0,NaN


## 5. 分析に使用する変数の確認
仮説検証に必要な変数が、応募案件マートに含まれていることを確認する。

In [169]:
required_columns = [
    "application_year",
    "recommended_flag",
    "withdrawal_flag",
    "process_withdrawal_flag",
    "same_occupation_experience_flag",
    "skill_gap",
    "wage_shortfall_rate",
    "wage_requirement_met_flag",
    "intent_to_recommend_days",
]

missing_columns = [col for col in required_columns if col not in application_df.columns]

assert not missing_columns, f"不足している列: {missing_columns}"

In [170]:
### 主要変数の欠損確認
application_df[
    [
        "recommended_flag",
        "same_occupation_experience_flag",
        "skill_gap",
    ]
].isna().sum()

recommended_flag                     0
same_occupation_experience_flag      0
skill_gap                          563
dtype: int64

In [171]:
### フラグ値の確認
print(application_df["recommended_flag"].value_counts(dropna=False))
print(application_df["same_occupation_experience_flag"].value_counts(dropna=False))

recommended_flag
1    1823
0     835
Name: count, dtype: int64
same_occupation_experience_flag
1    2095
0     563
Name: count, dtype: int64


## 6. 仮説1：スキル・経験ミスマッチと推薦率
EDAでは、2026年に同職種経験率の低下とスキルギャップの不足方向への変化が確認された。

そこで、求人が求める経験・スキルと応募者の保有経験・スキルのミスマッチが、推薦されにくさと関連しているかを確認する。

### 6.1 同職種経験の有無と推薦率
同職種経験の有無によって推薦率に違いがあるかを確認する。
まず、同職種経験あり・なしそれぞれについて、応募件数と推薦率を集計する。

"same_occupation_experience_flag"  
1 → 同職種経験有り  
0 → 同職種経験無し

In [172]:
experience_year_summary = (
    application_df
    .groupby(
        [
            "application_year",
            "same_occupation_experience_flag",
        ]
    )
    .agg(
        application_count=("recommended_flag", "count"),    # 件数
        recommendation_rate=("recommended_flag", "mean"),   # flagは0~1のため推薦の割合となる
    )
)

experience_year_summary

application_count  \
application_year same_occupation_experience_flag                      
2025             0                                              137   
                 1                                              940   
2026             0                                              426   
                 1                                             1155   

                                                  recommendation_rate  
application_year same_occupation_experience_flag                       
2025             0                                           0.291971  
                 1                                           0.865957  
2026             0                                           0.199531  
                 1                                           0.765368

### 6.2 スキルギャップと推薦率
推薦された応募者と推薦されなかった応募者について、スキルギャップの分布を比較する。
スキルギャップが不足方向に大きい応募者ほど推薦されにくい場合、推薦なしのグループでスキルギャップがより不足方向に偏ることが想定される。

"recommended_flag"  
1 → 推薦有り  
0 → 推薦無し

In [173]:
skill_gap_year_summary = (
    application_df
    .groupby(
        [
            "application_year",
            "recommended_flag",
        ]
    )["skill_gap"] # 各グループのskill_gapのみ集計
    .agg(
        [
            "count",    # 件数
            "mean",     # 平均
            "median",   # 中央値
        ]
    )
)

skill_gap_year_summary

count      mean  median
application_year recommended_flag                         
2025             0                   126 -0.269841     0.0
                 1                   814  0.028256     0.0
2026             0                   271 -0.236162     0.0
                 1                   884 -0.113122     0.0

### 6.3 統計的検定
#### 6.3.1 同職種経験の有無と推薦

同職種経験の有無と推薦結果に関連があるかを確認するため、カイ二乗検定を行う。

- 帰無仮説：同職種経験の有無と推薦結果に関連はない
- 対立仮説：同職種経験の有無と推薦結果に関連がある

In [174]:
experience_crosstab = pd.crosstab(
    application_df["same_occupation_experience_flag"],
    application_df["recommended_flag"],
)

experience_crosstab

recommended_flag,0,1
same_occupation_experience_flag,,
0,438,125
1,397,1698


In [175]:
# カイ二乗検定：独立性の検定
chi2, p_value, dof, expected = chi2_contingency(
    experience_crosstab
)

print(f"chi2 = {chi2:.3f}")             # カイ二乗統計量
print(f"p-value = {p_value:.4f}")       # p値
print(f"degrees of freedom = {dof}")    # 自由度

# "expected"は期待度数が極端に小さいセルのチェック使用する。今回は件数が十分に多いため出力しなくてOK

chi2 = 710.506
p-value = 0.0000
degrees of freedom = 1


In [176]:
# n：クロス集計表の全件数（分析対象の件数）
n = experience_crosstab.to_numpy().sum()

# カイ二乗統計量を使用して、Cramér's Vを計算
cramers_v = np.sqrt(
    chi2 / n
)

print(f"Cramer's V = {cramers_v:.3f}")

Cramer's V = 0.517


#### 6.3.2 スキルギャップと推薦
推薦された応募者と推薦されなかった応募者で、スキルギャップの分布に差があるかを確認する。
分布の偏りや外れ値の影響も考慮し、Mann-Whitney U検定を用いる。

- 帰無仮説：推薦あり・なしでスキルギャップの分布に差はない
- 対立仮説：推薦あり・なしでスキルギャップの分布に差がある

.groupby() → グループに分けて集計  
.loc[]　　 → 条件に合う行・列を抽出  
⇒Mann-Whitney U検定は2グループそれぞれの実データを渡す必要がある

In [177]:
recommended_skill_gap = (
    application_df
    .loc[
        application_df["recommended_flag"] == 1,    # 行の条件
        "skill_gap",    # 取得列
    ]
    .dropna()   # NULL除外
)

not_recommended_skill_gap = (
    application_df
    .loc[
        application_df["recommended_flag"] == 0,
        "skill_gap",
    ]
    .dropna()
)

In [178]:
# 推薦有・無の2つの独立したグループの分布に差があるかを確認する
u_stat, p_value = mannwhitneyu(
    recommended_skill_gap,
    not_recommended_skill_gap,
    alternative="two-sided",
)

n1 = len(recommended_skill_gap)         # 件数
n2 = len(not_recommended_skill_gap)     # 件数

# rank-biserial correlation（順位双列相関）で効果量を確認する
rank_biserial = (2 * u_stat) / (n1 * n2) - 1

print(f"U statistic = {u_stat:.3f}")    # U統計量
print(f"p-value = {p_value:.4f}")       # p値
print(f"Rank-biserial correlation = {rank_biserial:.3f}")

# rank_biserialの値の範囲は概ね（-1 ～ 1）
# 0に近い→差が小さい　±1に近い→差が大きい

U statistic = 387070.500
p-value = 0.0000
Rank-biserial correlation = 0.148


### 6.4 追加分析1：年別の同職種経験と推薦
2025年・2026年それぞれについて、同職種経験の有無と推薦有無をカイ二乗検定で確認し、Cramer's Vで効果量を算出

In [179]:
experience_test_results = []

for year in ["2025", "2026"]:   # 2025/2026に対してループ処理
    year_df = application_df[
        application_df["application_year"] == year
    ]

# クロス集計表：同職種経験の有無 × 推薦有無
    crosstab = pd.crosstab(
        year_df["same_occupation_experience_flag"],
        year_df["recommended_flag"],
    )

    chi2, p_value, dof, expected = chi2_contingency(crosstab)

    n = crosstab.to_numpy().sum()
    cramers_v = np.sqrt(chi2 / n)

# 検定結果を辞書形式でリストに追加
    experience_test_results.append({
        "year": year,
        "chi2": chi2,
        "p_value": p_value,
        "cramers_v": cramers_v,
    })

experience_test_results_df = pd.DataFrame(
    experience_test_results
)

experience_test_results_df

,year,chi2,p_value,cramers_v
0,2025,236.458018,2.328301e-53,0.468564
1,2026,417.601350,8.119781e-93,0.513943


### 6.5 追加分析2：年別のスキルギャップと推薦
2025年・2026年それぞれについて、推薦有無とスキルギャップの差をMann-Whitney U検定で確認し、rank-biserial correlationで効果量を算出

In [180]:
skill_gap_test_results = []

for year in ["2025", "2026"]:
    year_df = application_df[
        application_df["application_year"] == year
    ]

    recommended = (
        year_df
        .loc[
            year_df["recommended_flag"] == 1,
            "skill_gap",
        ]
        .dropna()
    )

    not_recommended = (
        year_df
        .loc[
            year_df["recommended_flag"] == 0,
            "skill_gap",
        ]
        .dropna()
    )

    u_stat, p_value = mannwhitneyu(
        recommended,
        not_recommended,
        alternative="two-sided",
    )

    n1 = len(recommended)
    n2 = len(not_recommended)

    rank_biserial = (
        (2 * u_stat) / (n1 * n2) - 1
    )

    skill_gap_test_results.append({
        "year": year,
        "u_statistic": u_stat,
        "p_value": p_value,
        "rank_biserial": rank_biserial,
    })

skill_gap_test_results_df = pd.DataFrame(
    skill_gap_test_results
)

skill_gap_test_results_df

,year,u_statistic,p_value,rank_biserial
0,2025,62193.0,0.000009,0.212765
1,2026,131067.0,0.006924,0.094213


### 6.6 結果整理

#### 同職種経験の有無は推薦結果と大きく関連

同職種経験の有無によって、推薦率には大きな差が確認された。

- 2025年：経験あり 86.6% / 経験なし 29.2%
- 2026年：経験あり 76.5% / 経験なし 20.0%

年別にカイ二乗検定を行った結果、2025年・2026年ともに
同職種経験の有無と推薦結果には統計的に有意な関連が確認された（いずれも p < 0.001）。

また、Cramer's Vは2025年が0.469、2026年が0.514であり、両年とも関連の大きさは比較的大きかった。

#### スキルギャップとの関連はあるが、差は限定的

スキルギャップについても、推薦あり・なしの間で統計的に有意な差が確認された。
一方、効果量を示すrank-biserial correlationは、2025年が0.213、2026年が0.094であった。

特に2026年では効果量が小さく、スキルギャップ単独による推薦結果の差は限定的と考えられる。

#### 仮説1の評価

以上から、「スキル・経験のミスマッチが大きいほど推薦されにくい」という仮説は支持される結果となった。
ただし、推薦結果との関連は一様ではなく、スキルギャップよりも同職種経験の有無との関連が大きいことが確認された。

#### 2026年の推薦率低下への示唆

2026年は同職種経験のない応募者が増加しており、この応募者構成の変化が全体の推薦率低下に関係している可能性がある。
一方で、同職種経験あり・なしのどちらのグループでも2026年は2025年より推薦率が低下している。

そのため、スキル・経験ミスマッチは推薦率低下に関連する要因の一つと考えられるが、
これだけでは2026年の推薦率低下をすべて説明できない。

次の仮説では、別の候補要因として給与条件のミスマッチを検証する。

## 7. 仮説2：給与条件のミスマッチ
EDAでは、2026年に給与不足率が上昇し、給与要件充足率(wage_requirement_met_flag)も低下していることを確認された。

そこで、応募者の希望給与が求人給与を上回る給与ミスマッチが、選考途中の辞退と関連しているかを確認する。

また、補足分析として、給与条件と推薦結果の関係についても確認する。

### 7.1 給与条件と選考途中辞退率
給与要件を満たしている応募者と満たしていない応募者で、選考途中の辞退率に違いがあるかを年別に確認する。

"wage_requirement_met_flag" → 応募者の希望給与を求人給与が満たしているかのフラグ  
"process_withdrawal_flag" → 選考途中辞退：accepted_other_job（他案件の就業決定で並行終了した応募案件）を除いた辞退

In [181]:
wage_withdrawal_summary = (
    application_df
    .groupby(
        [
            "application_year",
            "wage_requirement_met_flag",
        ]
    )
    .agg(
        application_count=("process_withdrawal_flag", "count"),
        withdrawal_rate=("process_withdrawal_flag", "mean"),
    )
)

wage_withdrawal_summary

application_count  withdrawal_rate
application_year wage_requirement_met_flag                                    
2025             0                                        500         0.182000
                 1                                        577         0.150780
2026             0                                        808         0.301980
                 1                                        773         0.190168

### 7.2 給与要件の充足と選考途中辞退
給与要件を満たしているかどうかと選考途中の辞退に関連があるかを確認するため、カイ二乗検定を行う。

- 帰無仮説：給与要件の充足と選考途中辞退に関連はない
- 対立仮説：給与要件の充足と選考途中辞退に関連がある

In [182]:
wage_withdrawal_crosstab = pd.crosstab(
    application_df["wage_requirement_met_flag"],
    application_df["process_withdrawal_flag"],
)

wage_withdrawal_crosstab

process_withdrawal_flag,0,1
wage_requirement_met_flag,,
0,973,335
1,1116,234


In [183]:
chi2, p_value, dof, expected = chi2_contingency(
    wage_withdrawal_crosstab
)

n = wage_withdrawal_crosstab.to_numpy().sum()
cramers_v = np.sqrt(chi2 / n)

print(f"chi2 = {chi2:.3f}")
print(f"p-value = {p_value:.4f}")
print(f"Cramer's V = {cramers_v:.3f}")

chi2 = 26.570
p-value = 0.0000
Cramer's V = 0.100


### 7.3 補足分析：給与条件と推薦率
給与条件が推薦判断とも関連している可能性があるため、補足として、給与要件を満たしている応募者と満たしていない応募者で、推薦率に違いがあるかを年別に確認する。

In [184]:
wage_recommend_summary = (
    application_df
    .groupby(
        [
            "application_year",
            "wage_requirement_met_flag",
        ]
    )
    .agg(
        application_count=("recommended_flag", "count"),
        recommendation_rate=("recommended_flag", "mean"),
    )
)

wage_recommend_summary

application_count  \
application_year wage_requirement_met_flag                      
2025             0                                        500   
                 1                                        577   
2026             0                                        808   
                 1                                        773   

                                            recommendation_rate  
application_year wage_requirement_met_flag                       
2025             0                                     0.826000  
                 1                                     0.764298  
2026             0                                     0.647277  
                 1                                     0.576973

### 7.4 補足分析：給与要件の充足と推薦

給与要件を満たしているかどうかと推薦結果に関連があるかを確認するため、
カイ二乗検定を行う。

- 帰無仮説：給与要件の充足と推薦結果に関連はない
- 対立仮説：給与要件の充足と推薦結果に関連がある

In [185]:
wage_recommend_crosstab = pd.crosstab(
    application_df["wage_requirement_met_flag"],
    application_df["recommended_flag"],
)

wage_recommend_crosstab

recommended_flag,0,1
wage_requirement_met_flag,,
0,372,936
1,463,887


In [186]:
chi2, p_value, dof, expected = chi2_contingency(
    wage_recommend_crosstab
)

n = wage_recommend_crosstab.to_numpy().sum()
cramers_v = np.sqrt(chi2 / n)

print(f"chi2 = {chi2:.3f}")
print(f"p-value = {p_value:.4f}")
print(f"Cramer's V = {cramers_v:.3f}")

chi2 = 10.303
p-value = 0.0013
Cramer's V = 0.062


### 7.5 追加分析：給与不足の程度と選考結果
ここまでの分析では、給与要件を満たしているかどうかを二値で比較した。

さらに、給与要件を満たしていないかどうかだけでなく、給与不足率が高くなるほど選考途中辞退や推薦結果に違いが生じるかを確認する。

給与不足率 → 希望給与に対する求人提示額の不足率 定義：MAX(希望時給 - 提示時給,0)

#### 7.5.1 辞退有無別の給与不足率
選考途中辞退あり・なしで、給与不足率の件数・平均値・中央値を比較する。

In [187]:
wage_shortfall_withdrawal_summary = (
    application_df
    .groupby("process_withdrawal_flag")["wage_shortfall_rate"]
    .agg(
        [
            "count",
            "mean",
            "median",
        ]
    )
)

wage_shortfall_withdrawal_summary

,count,mean,median
process_withdrawal_flag,,,
0,2089,0.056302,0.000
1,569,0.074984,0.045


### 7.5.2 辞退有無別の給与不足率（Mann-Whitney U検定）

選考途中辞退あり・なしで、給与不足率の分布に差があるかを確認する。
給与不足率は分布の偏りが考えられるため、Mann-Whitney U検定を用いる。

- 帰無仮説：辞退あり・なしで給与不足率の分布に差はない
- 対立仮説：辞退あり・なしで給与不足率の分布に差がある

In [188]:
withdrawal_wage_shortfall = (
    application_df
    .loc[
        application_df["process_withdrawal_flag"] == 1,
        "wage_shortfall_rate",
    ]
    .dropna()
)

no_withdrawal_wage_shortfall = (
    application_df
    .loc[
        application_df["process_withdrawal_flag"] == 0,
        "wage_shortfall_rate",
    ]
    .dropna()
)

u_stat, p_value = mannwhitneyu(
    withdrawal_wage_shortfall,
    no_withdrawal_wage_shortfall,
    alternative="two-sided",
)

n1 = len(withdrawal_wage_shortfall)
n2 = len(no_withdrawal_wage_shortfall)

rank_biserial = (2 * u_stat) / (n1 * n2) - 1

print(f"U statistic = {u_stat:.3f}")
print(f"p-value = {p_value:.4f}")
print(f"Rank-biserial correlation = {rank_biserial:.3f}")

U statistic = 688428.000
p-value = 0.0000
Rank-biserial correlation = 0.158


#### 7.5.3 給与不足率と推薦結果
補足として、推薦あり・なしでも給与不足率の件数、平均値、中央値を確認する。

In [189]:
wage_shortfall_summary = (
    application_df
    .groupby("recommended_flag")["wage_shortfall_rate"]
    .agg(
        [
            "count",
            "mean",
            "median",
        ]
    )
)

wage_shortfall_summary

,count,mean,median
recommended_flag,,,
0,835,0.071647,0.000
1,1823,0.055105,0.022


#### 7.5.4 給与不足率と推薦結果（Mann-Whitney U検定）

推薦された応募者と推薦されなかった応募者で、
給与不足率の分布に差があるかを確認する。

給与不足率は分布の偏りが考えられるため、
Mann-Whitney U検定を用いる。

- 帰無仮説：推薦あり・なしで給与不足率の分布に差はない
- 対立仮説：推薦あり・なしで給与不足率の分布に差がある

In [190]:
recommended_wage_shortfall = (
    application_df
    .loc[
        application_df["recommended_flag"] == 1,
        "wage_shortfall_rate",
    ]
    .dropna()
)

not_recommended_wage_shortfall = (
    application_df
    .loc[
        application_df["recommended_flag"] == 0,
        "wage_shortfall_rate",
    ]
    .dropna()
)

u_stat, p_value = mannwhitneyu(
    recommended_wage_shortfall,
    not_recommended_wage_shortfall,
    alternative="two-sided",
)

n1 = len(recommended_wage_shortfall)
n2 = len(not_recommended_wage_shortfall)

rank_biserial = (2 * u_stat) / (n1 * n2) - 1

print(f"U statistic = {u_stat:.3f}")
print(f"p-value = {p_value:.4f}")
print(f"Rank-biserial correlation = {rank_biserial:.3f}")

U statistic = 773684.500
p-value = 0.4624
Rank-biserial correlation = 0.017


### 7.6 結果整理

#### 給与条件と選考途中辞退の関連

選考途中辞退率は、両年とも応募者の希望給与を求人給与が満たしていない応募の方が高かった。
- 2025年：未充足 18.2% / 充足 15.1%
- 2026年：未充足 30.2% / 充足 19.0%

カイ二乗検定でも、給与要件の充足と選考途中辞退には
統計的に有意な関連が確認された（p < 0.001）。
ただし、Cramer's Vは0.100であり、関連の大きさは小さかった。

さらに、給与不足率を辞退あり・なしで比較した結果、
Mann-Whitney U検定でも統計的に有意な差が確認された（p < 0.001）。

rank-biserial correlationは0.158であり、給与不足率（希望給与に対する求人給与の不足率）が高い応募ほど辞退する傾向は見られるものの、
その差の大きさは小さかった。


#### 補足：給与条件と推薦結果の関連は限定的

給与要件の充足状況別に推薦率を確認すると、2025年・2026年ともに、
応募者の希望給与を求人給与が満たしていない応募の方が推薦率はわずかに高かった。

カイ二乗検定では、給与要件の充足と推薦結果に統計的に有意な関連が確認された（p = 0.0013）。
一方、Cramer's Vは0.062と非常に小さく、関連の大きさは限定的であった。

以上から、給与条件のミスマッチが「推薦されにくさ」につながっているという明確な傾向は確認されなかった。

#### 仮説2の評価

以上から、**「応募者の希望給与が求人給与を上回ると、選考途中辞退が発生しやすいのではないか」**という仮説は支持される結果となった。

給与要件を満たしていない応募者では辞退率が高く、給与不足率そのものについても辞退との関連が確認された。

ただし、いずれも効果量は大きくないことから、給与条件だけで辞退を説明できるわけではなく、他の要因と組み合わさって影響している可能性がある。

一方、補足として確認した推薦結果については、給与条件のミスマッチによって推薦されにくくなるという傾向は確認されなかった。

#### 2026年の辞退率悪化への示唆

2026年は選考途中辞退率が上昇しており、特に希望給与が求人給与を上回る応募では30.2%まで上昇している。

そのため、給与ミスマッチは2026年の選考途中の離脱増加に一定程度関係している可能性がある。

一方、2026年は給与要件の充足・未充足にかかわらず推薦率が低下しているため、給与条件だけでは推薦率低下を説明できない。

次の仮説では、選考リードタイムの長期化と辞退の関係を検証する。

## 8. 仮説3：選考リードタイムと辞退
EDAでは、応募意思確認から推薦までの日数が
2025年の平均約1.5日から2026年は約6.0日へ長期化していた。

そこで、応募意思確認から推薦までのリードタイムが長いほど、選考途中辞退が発生しやすいかを確認する。

In [191]:
lead_time_df = (
    application_df
    .dropna(
        subset=[
            "intent_to_recommend_days",
            "process_withdrawal_flag",
        ]
    )
    .copy()
)

### 8.1 辞退有無によるリードタイム比較
選考途中辞退あり・なしで、応募意思確認から推薦までのリードタイムに違いがあるかを確認する。

集計対象 → 応募意思確認から推薦までのリードタイム

In [192]:
lead_time_summary = (
    lead_time_df
    .groupby(
        [
            "application_year",
            "process_withdrawal_flag",
        ]
    )["intent_to_recommend_days"]
    .agg(
        [
            "count",
            "mean",
            "median",
        ]
    )
)

lead_time_summary

count      mean  median
application_year process_withdrawal_flag                         
2025             0                          722  2.065097     2.0
                 1                          132  2.333333     2.0
2026             0                          780  5.728205     6.0
                 1                          189  5.703704     6.0

### 8.2 リードタイム別の辞退率
応募意思確認から推薦までの日数を区分し、リードタイムが長くなるにつれて選考途中辞退率が上昇するかを確認する。

In [193]:
lead_time_df["lead_time_group"] = pd.cut(
    lead_time_df["intent_to_recommend_days"],
    bins=[-1, 1, 3, 7, np.inf], # 区切り位置の指定（右端を含む区間）
    labels=[
        "0-1 days",
        "2-3 days",
        "4-7 days",
        "8+ days",
    ],
)

lead_time_withdrawal_summary = (
    lead_time_df
    .groupby(
        "lead_time_group",
        observed=True,
    )
    .agg(
        application_count=("process_withdrawal_flag", "count"),
        withdrawal_rate=("process_withdrawal_flag", "mean"),
    )
)

lead_time_withdrawal_summary

,application_count,withdrawal_rate
lead_time_group,,
0-1 days,355,0.146479
2-3 days,430,0.153488
4-7 days,953,0.196222
8+ days,85,0.188235


### 8.3 統計的検定
選考途中辞退あり・なしでリードタイムの分布に差があるかを確認するため、Mann-Whitney U検定を行う。

- 帰無仮説：辞退あり・なしでリードタイムの分布に差はない
- 対立仮説：辞退あり・なしでリードタイムの分布に差がある

あわせて、差の大きさを確認するため、
rank-biserial correlationを算出する。

In [194]:
withdrawal_lead_time = (
    lead_time_df
    .loc[
        lead_time_df["process_withdrawal_flag"] == 1,
        "intent_to_recommend_days",
    ]
)

no_withdrawal_lead_time = (
    lead_time_df
    .loc[
        lead_time_df["process_withdrawal_flag"] == 0,
        "intent_to_recommend_days",
    ]
)

u_stat, p_value = mannwhitneyu(
    withdrawal_lead_time,
    no_withdrawal_lead_time,
    alternative="two-sided",
)

n1 = len(withdrawal_lead_time)
n2 = len(no_withdrawal_lead_time)

rank_biserial = (2 * u_stat) / (n1 * n2) - 1

print(f"U statistic = {u_stat:.3f}")
print(f"p-value = {p_value:.4f}")
print(f"Rank-biserial correlation = {rank_biserial:.3f}")

U statistic = 260846.000
p-value = 0.0198
Rank-biserial correlation = 0.082


#### 8.3.1 追加分析1：年別のMann-Whitney U検定
リードタイムが長いほど辞退率がやや高いように見えるが、年別の平均・中央値およびMann-Whitney U検定の結果から関係がかなり弱そう。
そのため年別で追加分析を実施する。

In [195]:
lead_time_test_results = []

for year in ["2025", "2026"]:
    year_df = lead_time_df[
        lead_time_df["application_year"] == year
    ]

    withdrawal = (
        year_df
        .loc[
            year_df["process_withdrawal_flag"] == 1,
            "intent_to_recommend_days",
        ]
        .dropna()
    )

    no_withdrawal = (
        year_df
        .loc[
            year_df["process_withdrawal_flag"] == 0,
            "intent_to_recommend_days",
        ]
        .dropna()
    )

    u_stat, p_value = mannwhitneyu(
        withdrawal,
        no_withdrawal,
        alternative="two-sided",
    )

    n1 = len(withdrawal)
    n2 = len(no_withdrawal)

    rank_biserial = (
        (2 * u_stat) / (n1 * n2) - 1
    )

    lead_time_test_results.append({
        "year": year,
        "u_statistic": u_stat,
        "p_value": p_value,
        "rank_biserial": rank_biserial,
    })

lead_time_test_results_df = pd.DataFrame(
    lead_time_test_results
)

lead_time_test_results_df

,year,u_statistic,p_value,rank_biserial
0,2025,51480.5,0.132256,0.080343
1,2026,72070.5,0.626038,-0.022243


### 8.3.2 追加分析2：リードタイム別辞退率も年別に見る

In [196]:
lead_time_withdrawal_year_summary = (
    lead_time_df
    .groupby(
        [
            "application_year",
            "lead_time_group",
        ],
        observed=True,  # 年 × リードタイム区分のうち、実際に応募データが存在する組み合わせだけ集計する
    )
    .agg(
        application_count=("process_withdrawal_flag", "count"),
        withdrawal_rate=("process_withdrawal_flag", "mean"),
    )
)

lead_time_withdrawal_year_summary

application_count  withdrawal_rate
application_year lead_time_group                                    
2025             0-1 days                       346         0.147399
                 2-3 days                       357         0.142857
                 4-7 days                       149         0.201342
                 8+ days                          2         0.000000
2026             0-1 days                         9         0.111111
                 2-3 days                        73         0.205479
                 4-7 days                       804         0.195274
                 8+ days                         83         0.192771

### 8.4 結果整理

#### 全期間ではリードタイムと辞退に差が見られた

全期間を対象に、選考途中辞退あり・なしで応募意思確認から推薦までのリードタイムを比較した。
Mann-Whitney U検定では統計的に有意な差が確認された（p = 0.0198）。

一方、rank-biserial correlationは0.082であり、差の大きさは非常に小さかった。


#### 年別ではリードタイムと辞退の明確な関連は確認されなかった

2025年・2026年に分けて同様の検定を行った結果、
いずれの年でも統計的に有意な差は確認されなかった。

- 2025年：p = 0.132、rank-biserial correlation = 0.080
- 2026年：p = 0.626、rank-biserial correlation = -0.022

特に2026年では、辞退なしの平均リードタイムが5.73日、辞退ありが5.70日であり、
中央値もいずれも6日とほぼ同じであった。

また、リードタイム帯別の辞退率を年別に確認しても、
リードタイムが長くなるほど辞退率が一貫して上昇する傾向は見られなかった。


#### 仮説3の評価

以上から、「応募意思確認から推薦までのリードタイムが長いほど、
選考途中辞退が発生しやすい」という仮説は支持されなかった。

全期間では統計的な差が見られたものの、
2025年と2026年ではリードタイムの水準そのものが大きく異なるため、
年次差の影響を受けた結果である可能性が高い。


#### 2026年の辞退率上昇への示唆

2026年は2025年と比較してリードタイムが大きく長期化している一方、
同じ2026年の中ではリードタイムの長さと辞退との明確な関連は確認されなかった。

そのため、リードタイム長期化は2026年に発生した大きな変化ではあるものの、
それ自体が選考途中辞退の増加を説明する主要因とは考えにくい。

2026年の辞退率上昇については、
給与条件など他の要因や複数要因の組み合わせを考慮する必要がある。

## 9. 複数要因を考慮した分析
これまでの分析では、各要因と推薦・辞退との関係を個別に確認した。

ここではロジスティック回帰を用いて複数要因を同時に考慮し、他の条件を調整した場合でも各要因と推薦・辞退に関連があるかを確認する。

### 9.1 推薦に関連する要因
推薦有無を目的変数とし、同職種経験、給与不足率、年次を説明変数としたロジスティック回帰を行う。

smf.logit(...)はロジスティック回帰を行う関数  
ロジスティック回帰は2値データの時に使用する  

In [197]:
recommend_model_df = (
    application_df[
        [
            "recommended_flag",
            "same_occupation_experience_flag",
            "wage_shortfall_rate",
            "application_year",
        ]
    ]
    .dropna()
    .copy()
)

recommend_model_df.shape

(2658, 4)

In [198]:
recommend_model = smf.logit(
    """
    recommended_flag
    ~ same_occupation_experience_flag
    + wage_shortfall_rate
    + C(application_year)
    """,
    data=recommend_model_df,
).fit(disp=False)

recommend_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:       recommended_flag   No. Observations:                 2658
Model:                          Logit   Df Residuals:                     2654
Method:                           MLE   Df Model:                            3
Date:                Tue, 06 Oct 2026   Pseudo R-squ.:                  0.2170
Time:                        00:56:38   Log-Likelihood:                -1295.3
converged:                       True   LL-Null:                       -1654.3
Covariance Type:            nonrobust   LLR p-value:                2.749e-155
===================================================================================================
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                          -0.7677      0.129     -5.939      0.000      -1.021      -0.514
C(application_year)[T.2026]        -0.6389      0.104     -6.129      0.000      -0.843      -0.435
same_occupation_experience_flag     2.6211      0.117     22.381      0.000       2.392       2.851
wage_shortfall_rate                -0.2967      0.503     -0.590      0.555      -1.282       0.688
===================================================================================================
"""

In [199]:
recommend_odds_ratio = pd.DataFrame({
    "odds_ratio": np.exp(recommend_model.params),
    "p_value": recommend_model.pvalues,
})

recommend_odds_ratio

,odds_ratio,p_value
Intercept,0.464098,2.874878e-09
C(application_year)[T.2026],0.527856,8.847979e-10
same_occupation_experience_flag,13.750923,5.978487e-111
wage_shortfall_rate,0.743298,5.550247e-01


### 9.2 辞退に関連する要因
選考途中辞退については、
応募意思確認から推薦までのリードタイムが算出できる案件のみを対象に分析した。

確認の結果、分析対象1823件はすべて推薦済み案件であり、同職種経験ありの割合も93.1%と高かった。

そのため、本モデルは全応募案件ではなく、
「推薦まで到達した案件における辞退要因」を確認する分析として解釈する。

In [200]:
withdrawal_model_df = (
    application_df[
        [
            "process_withdrawal_flag",
            "same_occupation_experience_flag",
            "wage_shortfall_rate",
            "intent_to_recommend_days",
            "application_year",
        ]
    ]
    .dropna()
    .copy()
)

withdrawal_model_df.shape

(1823, 5)

In [201]:
withdrawal_model = smf.logit(
    """
    process_withdrawal_flag
    ~ same_occupation_experience_flag
    + wage_shortfall_rate
    + intent_to_recommend_days
    + C(application_year)
    """,
    data=withdrawal_model_df,
).fit(disp=False)

withdrawal_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                              Logit Regression Results                             
===================================================================================
Dep. Variable:     process_withdrawal_flag   No. Observations:                 1823
Model:                               Logit   Df Residuals:                     1818
Method:                                MLE   Df Model:                            4
Date:                     Tue, 06 Oct 2026   Pseudo R-squ.:                 0.01623
Time:                             00:56:38   Log-Likelihood:                -834.65
converged:                            True   LL-Null:                       -848.43
Covariance Type:                 nonrobust   LLR p-value:                 1.541e-05
===================================================================================================
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                          -2.9455      0.360     -8.172      0.000      -3.652      -2.239
C(application_year)[T.2026]         0.1029      0.195      0.528      0.597      -0.279       0.485
same_occupation_experience_flag     1.0521      0.333      3.157      0.002       0.399       1.705
wage_shortfall_rate                 2.4669      0.696      3.545      0.000       1.103       3.831
intent_to_recommend_days            0.0483      0.041      1.174      0.241      -0.032       0.129
===================================================================================================
"""

In [202]:
withdrawal_odds_ratio = pd.DataFrame({
    "odds_ratio": np.exp(withdrawal_model.params),
    "p_value": withdrawal_model.pvalues,
})

withdrawal_odds_ratio

,odds_ratio,p_value
Intercept,0.052574,3.040799e-16
C(application_year)[T.2026],1.108405,5.973354e-01
same_occupation_experience_flag,2.863551,1.595382e-03
wage_shortfall_rate,11.785562,3.919774e-04
intent_to_recommend_days,1.049472,2.405636e-01


In [203]:
withdrawal_model_check = (
    application_df
    .assign(
        included_in_withdrawal_model=
        application_df[
            [
                "process_withdrawal_flag",
                "same_occupation_experience_flag",
                "wage_shortfall_rate",
                "intent_to_recommend_days",
                "application_year",
            ]
        ]
        .notna()
        .all(axis=1)
    )
    .groupby("included_in_withdrawal_model")
    .agg(
        application_count=("recommended_flag", "count"),
        same_occupation_experience_rate=(
            "same_occupation_experience_flag",
            "mean",
        ),
        withdrawal_rate=(
            "process_withdrawal_flag",
            "mean",
        ),
        recommendation_rate=(
            "recommended_flag",
            "mean",
        ),
    )
)

withdrawal_model_check

,application_count,same_occupation_experience_rate,withdrawal_rate,recommendation_rate
included_in_withdrawal_model,,,,
False,835,0.475449,0.297006,0.0
True,1823,0.931432,0.176083,1.0


In [204]:
recommended_same_exp_df = (
    application_df
    .loc[
        (application_df["recommended_flag"] == 1)
        & (application_df["same_occupation_experience_flag"] == 1)
    ]
    .dropna(
        subset=[
            "skill_gap",
            "process_withdrawal_flag",
        ]
    )
    .copy()
)

recommended_same_exp_summary = (
    recommended_same_exp_df
    .groupby("process_withdrawal_flag")["skill_gap"]
    .agg(
        [
            "count",
            "mean",
            "median",
        ]
    )
)

recommended_same_exp_summary

,count,mean,median
process_withdrawal_flag,,,
0,1388,-0.063401,0.0
1,310,0.035484,0.0


In [205]:
withdrawal_skill_gap = (
    recommended_same_exp_df
    .loc[
        recommended_same_exp_df["process_withdrawal_flag"] == 1,
        "skill_gap",
    ]
)

no_withdrawal_skill_gap = (
    recommended_same_exp_df
    .loc[
        recommended_same_exp_df["process_withdrawal_flag"] == 0,
        "skill_gap",
    ]
)

u_stat, p_value = mannwhitneyu(
    withdrawal_skill_gap,
    no_withdrawal_skill_gap,
    alternative="two-sided",
)

n1 = len(withdrawal_skill_gap)
n2 = len(no_withdrawal_skill_gap)

rank_biserial = (2 * u_stat) / (n1 * n2) - 1

print(f"U statistic = {u_stat:.3f}")
print(f"p-value = {p_value:.4f}")
print(f"Rank-biserial correlation = {rank_biserial:.3f}")

U statistic = 231670.500
p-value = 0.0128
Rank-biserial correlation = 0.077


## 10. 仮説検証まとめ

EDAで確認した2026年の推薦率低下や選考途中辞退の増加について、
スキル・経験のミスマッチ、給与条件のミスマッチ、
選考リードタイムの3つの観点から検証した。

### 1. 推薦には「同職種経験」の有無が大きく関連

同職種経験の有無と推薦結果には、2025年・2026年ともに統計的に有意な関連が確認された。

Cramer's Vも2025年が0.469、2026年が0.514と比較的大きく、
多変量分析でも、同職種経験ありの応募者は経験なしと比べて推薦オッズが約13.8倍であった。

一方、スキルギャップについては統計的な差は確認されたものの、
効果量は小さく、特に2026年では実質的な差は限定的であった。

以上から、推薦判断では細かなスキル差よりも、
「同職種経験があるか」という経験属性の方が強く関係していることが確認された。


### 2. 給与ミスマッチは選考途中辞退と関連

給与要件を満たしていない応募者では選考途中辞退率が高く、
給与不足率についても辞退あり・なしで統計的に有意な差が確認された。

多変量分析でも、給与不足率は他の要因を考慮した後も
選考途中辞退と有意に関連していた。

一方、補足として推薦結果との関係も確認したが、
給与要件の充足と推薦結果の関連は小さく、
給与不足率についても推薦あり・なしで有意な差は確認されなかった。

以上から、給与条件のミスマッチは推薦判断よりも、
選考途中辞退に関係する要因と考えられる。


### 3. リードタイム長期化と辞退の明確な関連は確認されなかった

EDAでは、2026年に応募意思確認から推薦までのリードタイムが
大きく長期化していたため、辞退増加との関連を仮説として検証した。

全期間では辞退あり・なしで統計的な差が確認されたものの、効果量は非常に小さかった。

さらに年別に分析すると、2025年・2026年ともに有意な差は確認されず、
リードタイムが長くなるほど辞退率が一貫して上昇する傾向も見られなかった。
多変量分析でもリードタイムと辞退の有意な関連は確認されなかった。

そのため、2026年にリードタイムと辞退率が同時に上昇していたものの、
リードタイム長期化そのものが辞退増加の主要因とは考えにくい。


### 4. 2026年の推薦率・辞退率悪化への示唆

2026年の推薦率低下には、
同職種経験のない応募者が増加したことが関係している可能性が高い。

一方で、同職種経験あり・なしのどちらのグループでも2026年は推薦率が低下しており、
多変量分析でも2026年は2025年より推薦されにくい傾向が残った。

そのため、同職種経験の構成変化だけでは2026年の推薦率低下をすべて説明することはできない。

また、選考途中辞退については給与条件との関連が確認された一方、リードタイムとの明確な関連は確認されなかった。

以上から、2026年の選考前半の悪化は、単一の要因ではなく、
- 同職種経験のない応募者の増加
- 給与条件のミスマッチによる選考途中辞退
- 今回の分析では捉えきれていないその他の要因
が組み合わさって生じている可能性がある。


### 仮説検証結果

- 仮説1：スキル・経験ミスマッチ → 支持
  - 特に同職種経験の有無との関連が大きい
- 仮説2：給与条件のミスマッチ → 支持
  - 給与条件のミスマッチと選考途中辞退の関連が確認された
- 仮説3：選考リードタイム → 支持されず
  - 年次差を考慮すると、辞退との明確な関連は確認されなかった

## 11. 事業課題への示唆・改善施策

今回の分析では、2026年の求人充足率低下について、
推薦率の低下と選考途中辞退の増加に関連する要因を検証した。

分析結果を踏まえ、以下の改善施策を検討する。

### 1. 重点職種における経験者の母集団形成を強化する

同職種経験の有無は推薦結果と強く関連しており、2026年は同職種経験のない応募者が増加していた。

そのため、求人需要の大きい職種を重点職種として設定し、
該当職種の経験者を増やすための集客・スカウト施策を強化することが有効と考える。

具体的には、求人媒体やスカウト、登録導線において
重点職種ごとに訴求内容を分け、求人需要に対して必要な経験を持つ候補者の母集団形成を強化する。

### 2. 給与条件のミスマッチを推薦前に把握・調整する

給与条件を満たしていない応募者では選考途中辞退率が高く、給与不足率についても選考途中辞退との関連が確認された。

そのため、推薦前に応募者の希望給与と求人給与の乖離を把握し、
乖離が大きい案件については、候補者への条件確認や派遣先との条件交渉を早期に行うことが有効と考える。

今後は給与不足率別の辞退率を確認し、辞退リスクが高まる水準を把握することで、
条件交渉を優先する案件の基準設定につなげる。

### 3. リードタイム短縮は最優先施策としない

2026年は応募意思確認から推薦までのリードタイムが大きく長期化していたが、
年次差を考慮した分析では、リードタイムと選考途中辞退の明確な関連は確認されなかった。

そのため、リードタイム短縮を最優先の改善施策とするよりも、
まずは「重点職種の経験者獲得」と「給与条件のミスマッチ解消」を優先することが妥当と考える。